# CFL sweep follow-up for linear advection
This notebook asks one tighter question than the earlier scheme cards. If the timestep moves toward a one-cell translation, how much of the old blur-versus-ringing hierarchy is still visible?


In [ ]:
from advectionlab.analysis import study_transport
requested_cfls = tuple(round(0.2 + 0.05 * index, 2) for index in range(17))
rows = study_transport(
    schemes=('upwind', 'lax-friedrichs', 'lax-wendroff', 'tvd-minmod'),
    requested_cfls=requested_cfls,
)
len(rows)


## Read the two lanes separately
The Gaussian lane stays smooth enough that Lax-Wendroff keeps first place all the way across. The square lane still rewards the limiter, but the gap starts collapsing as CFL approaches 1 because the update is getting closer to a sampled grid shift.


In [ ]:
for cfl in (0.4, 0.8, 0.95, 1.0):
    square_rows = [row for row in rows if row.profile_key == 'square' and abs(row.requested_cfl - cfl) < 1e-9]
    ranking = sorted(square_rows, key=lambda row: row.l2_error)
    print(cfl, [(row.scheme_key, round(row.l2_error, 4), round(row.overshoot, 4)) for row in ranking])


## Why the endpoint is exact here
At CFL 1 with positive velocity, each explicit update becomes an exact one-cell shift on this periodic grid. After one full turn the sampled profile comes back to itself, so every scheme lands at zero transport error. That is a geometric endpoint of this setup, not a general theorem that scheme choice suddenly stops mattering.


In [ ]:
unit_rows = [row for row in rows if abs(row.requested_cfl - 1.0) < 1e-9]
[(row.scheme_key, row.profile_key, row.l2_error, row.overshoot) for row in unit_rows]


## Adversarial check
If the exact endpoint were the only thing happening, the whole sweep would be a cheap trick. It is not. The non-unit-CFL part still shows the real ordering: Lax-Wendroff owns the smooth lane, TVD minmod owns the bounded jump lane, and the approach to CFL 1 only tells you when that ranking gets compressed by the geometry of the step itself.
